In [ ]:
import os
import numpy as np
import pandas as pd


hydro_file = 'your_directory/average_county_curves.npz'
mapping_file = 'your_directory/fips_to_subregion_mapping.csv'
output_dir = 'your_directory/hydro'
os.makedirs(output_dir, exist_ok=True)

county_hydro_data = np.load(hydro_file, allow_pickle=True)

mapping_df = pd.read_csv(mapping_file, dtype={0: str})  # 保证 county code 前导 0 不丢失
mapping_df.columns = ['county_code', 'unused', 'subregion_code']
county_to_subregion = mapping_df.set_index('county_code')['subregion_code'].to_dict()

subregion_hydro = {}

for county_code, hydro_curve in county_hydro_data.items():
    county_code = str(county_code).zfill(5)  # 补齐 5 位
    if county_code not in county_to_subregion:
        continue
    subregion = str(county_to_subregion[county_code])
    if subregion not in subregion_hydro:
        subregion_hydro[subregion] = np.zeros_like(hydro_curve, dtype=np.float32)
    subregion_hydro[subregion] += hydro_curve.astype(np.float32)*1.5

for subregion, hydro_curve in subregion_hydro.items():
    output_path = os.path.join(output_dir, f"subregion_{subregion}_hydro.npy")
    np.save(output_path, hydro_curve)
    print(f"Saved: {output_path}")

print("All subregion hydro curves have been saved.")


In [ ]:
import matplotlib.pyplot as plt

county_curves = np.load('your_directory/subregion_1_hydro.npy')  # shape: (8760,)

assert county_curves.shape[0] == 8760, "county_curve should have 8760 hourly values."

daily_data = county_curves.reshape(365, 24)  # shape: (365, 24)

mean_daily_curve = daily_data.mean(axis=0)  # shape: (24,)

plt.figure(figsize=(8, 4))
plt.plot(range(24), mean_daily_curve, marker='o')
plt.xticks(range(0, 24, 2))
plt.xlabel("Hour of Day")
plt.ylabel("Average Power (MW or Unit)")
plt.title("Average Daily Load Curve (Hydro, Subregion 1)")
plt.grid(True)
plt.tight_layout()
plt.show()
